In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/external-catlog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
from pyspark.sql.functions import col

# Filter high risk attrition employees
high_risk_df = df.filter((col("Attrition") == "No") & (col("JobSatisfaction") < 3))

# Select relevant informative columns
selected_df = high_risk_df.select(
    "EmployeeNumber", "Department", "JobRole", "JobSatisfaction", "Attrition", "Age", "MonthlyIncome"
)

# Write to Delta table in default schema of external-catlog
selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catlog`.default.high_risk_attrition_employees")

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "`external-catlog`.default.high_risk_attrition_employees")
history_df = deltaTable.history().select("version", "timestamp", "operation")
display(history_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
from pyspark.sql import Row

dummy_row = Row(
    EmployeeNumber=99999,
    Department="DummyDept",
    JobRole="DummyRole",
    JobSatisfaction=1,
    Attrition="No",
    Age=30,
    MonthlyIncome=0
)

dummy_df = spark.createDataFrame([dummy_row])
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catlog`.default.high_risk_attrition_employees")

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "`external-catlog`.default.high_risk_attrition_employees")
history_df = deltaTable.history().select("version", "timestamp", "operation")
display(history_df)

In [0]:
df_delta = spark.read.format("delta").table("`external-catlog`.default.high_risk_attrition_employees")
display(df_delta)

In [0]:
df_version = spark.read.option("versionAsOf", 0).table("`external-catlog`.default.high_risk_attrition_employees")
display(df_version)

In [0]:
df_timestamp = spark.read.option("timestampAsOf", "2025-12-06 11:30:00").table("`external-catlog`.default.high_risk_attrition_employees")
display(df_timestamp)

In [0]:
spark.sql("CR`external-catlog`EATE VOLUME `external-catlog`.default.employee_transformed_data")

In [0]:
from pyspark.sql.functions import col

# Logical transformation: filter employees with MonthlyIncome > 3000
transformed_df = df.filter(col("MonthlyIncome") > 3000)

# Write to volume partitioned by Department
transformed_df.write.partitionBy("Department").format("parquet").mode("overwrite").save("/Volumes/external-catlog/default/employee_transformed_data")